# L2-03 配套：基础模型路线的协议核算配套课文：[`docs/lessons/L2-03-基础模型路线.md`](../docs/lessons/L2-03-基础模型路线.md)。**本课回答四个能在纯 NumPy 里验证、不需要任何权重的问题：**1. scGPT 的参数量到底是多少——以及为什么**不能**给出一个「总参数量」；2. 当靶点在某背景中表达为零时，scGPT 的逐基因扰动接口还剩多少可区分信息；3. 「基础模型打不过线性基线」的机制：共享分量被 B1 吃掉、背景特异分量被噪声淹没；4. 本赛 300 个靶点该把算力投到哪里（transferable fraction 分档）。**边界（读完再往下看）：** 这里**没有**官方 scGPT 模型、没有权重、没有 torch，也**没有联网**。所有数字来自两类：(a) 固定 commit `cebd6fae65` 的源码与配置里的**超参字面值**；(b) 本 notebook 的**纯 NumPy 合成实验**。第 3、4 单元用的是**合成数据**，它演示的是**机制**（在什么条件下 B1 够用、在什么条件下高容量模型才有空间），**不是** scBaseCount 或任何真实数据集的实测结果。**耗时一个数字都没有。**

In [ ]:
# -*- coding: utf-8 -*-
"""单元 0：环境。纯 NumPy + matplotlib，不需要 torch / GPU / 权重。"""
import json
import math
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# 图内中文：Windows 上优先用系统 CJK 字体
_CJK = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC", "DengXian"]
_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _f in _CJK:
    if _f in _available:
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

rng = np.random.default_rng(20260918)

print("numpy   =", np.__version__)
print("字体    =", plt.rcParams["font.sans-serif"][0])
print("源码 commit = cebd6fae65 (bowang-lab/scGPT)")

## 单元 1｜scGPT 的分层参数量：为什么不能给一个总数课文 §3.2 说得很直接：**要引用 scGPT 的规模，引用它的层数与嵌入维度（12 层 / 512），不要引用一个来源不明的总参数。** 这一格把这句话算清楚。超参取自论文实现细节 `[P2]`，与 `tutorials/Tutorial_Perturbation.ipynb` 的实参一致 `[S1]`：`embsize=512, d_hid=512, nlayers=12, nhead=8`。**关键**：`TransformerModel.__init__` 里 `d_model / nhead / d_hid / nlayers` **全是必填无默认值**（`model.py:25-32`），唯一有默认值的是 `dropout=0.5`。所谓「默认超参」只活在脚本和教程里。

In [ ]:
# ---- 超参：来自论文实现细节 + 扰动教程实参，不是源码默认值 ----
d_model = 512          # embsize
d_hid = 512            # 前馈隐藏
nlayers = 12
nhead = 8
vocab_size = 48292     # 本地解析 scgpt/tokenizer/default_gene_vocab.json 实测值

def lin(i, o, bias=True):
    return i * o + (o if bias else 0)

# 逐层分解（全部按源码结构手算）
parts = {}
parts["值编码器 ContinuousValueEncoder"] = lin(1, d_model) + lin(d_model, d_model) + 2 * d_model
parts["每层 自注意力 (qkv+out)"] = 4 * lin(d_model, d_model, bias=False) + 4 * d_model
parts["每层 前馈 (Linear→Linear)"] = lin(d_model, d_hid) + lin(d_hid, d_model) + 2 * d_model
parts["每层 两个 LayerNorm"] = 4 * d_model
per_layer = parts["每层 自注意力 (qkv+out)"] + parts["每层 前馈 (Linear→Linear)"] + parts["每层 两个 LayerNorm"]

def show(label, value):
    """scGPT 参数量按千分位打印；label 定宽，避免中文对齐抖动。"""
    if isinstance(value, str):
        print(f"{label: <34} {value: >14}")
    else:
        print(f"{label: <34} {value: >14,}")

show("模块", "参数量")
print("-" * 50)
for k, v in parts.items():
    show(k, v)
print("-" * 50)
show("每层合计", per_layer)
show("TransformerEncoder (%d 层)" % nlayers, per_layer * nlayers)

dec1 = lin(d_model, d_model) + lin(d_model, d_model) + lin(d_model, 1)
dec = 2 * dec1
show("AffineExprDecoder（两个 ExprDecoder）", dec)
show("分类头 ClsDecoder (nlayers_cls=3)", 2 * (lin(d_model, d_model) + 2 * d_model) + lin(d_model, 1))
print()
print("--- 单列为一行、绝不并入总数的两项 ---")
show("基因嵌入 (48,292 × 512)", vocab_size * d_model)
print(f"{'基因嵌入说明': <34} {'← 取决于词表，工程实现还会加特殊 token': >14}")
show("MVC 头", "按开关")
print(f"{'MVC 说明': <34} {'← 是否启用 do_mvc / explicit_zero_prob 会再加': >14}")
print()
print("--- 只算「结构本体」的两个口径 ---")
body_no_embed = per_layer * nlayers + dec + parts["值编码器 ContinuousValueEncoder"]
cls_head = 2 * (lin(d_model, d_model) + 2 * d_model) + lin(d_model, 1)
show("不含基因嵌入与分类头", body_no_embed)
show("含分类头、不含基因嵌入", body_no_embed + cls_head)
print(f"{'口径说明': <34} {'← 对应论文的 12 层 / 512 维': >14}")
print()
print("对照：同族模型的公开参数量")
print("  State Base  76,740,712（L2-01 手算，与论文表 3 一致）")
print("  Stack Large 217,484,712（L2-02 手算，与论文表 3 一致）")
print()
print("结论：scGPT 的「结构本体」与 State/Stack 同量级；总参数由词表项主导，")
print("      所以引用规模时应引用 12 层 / 512 维，而不是一个总数。")

## 单元 2｜靶点表达为零时，逐基因扰动接口还剩多少信息课文 §3.4 提出一个关键判断：> scGPT 的 `Embedding(3, d)` 只能表达「某基因是否被扰动」；「被扰动的是哪个基因」是由> **该基因自身的位置**携带的。这一格把这句话变成一个**可数的量**。做法：构造一个极简的「模型可区分输入空间」——输入给模型的每个基因位置上有两个字段（表达值、扰动标记），模型能看见的就是这两个字段的组合。然后问：**当靶点在 NTC 中的表达值为 0 时，模型能从这个位置区分出多少个不同的靶点？**

In [ ]:
# ---- scGPT 在某个基因位置上的输入：两个字段 ----
# 表达值：连续，但 clamp(max=512)，且预训练只输入非零基因
# 扰动标记：Embedding(3, d_model)，取值 {0,1,2}（默认 pert_pad_id=2，教程设 0）
PERT_VALUES = 3          # Embedding(3, ...) 的字面值
CLAMP_MAX = 512          # ContinuousValueEncoder 的 max_value 默认值

def distinct_inputs(expr_value):
    """给定该基因的表达值，模型在该位置能看到的 (值, 标记) 组合数。"""
    return len({(round(expr_value, 6), p) for p in range(PERT_VALUES)})

print("%-30s %-16s %s" % ("该基因在 NTC 中的表达", "值字段", "模型可区分的输入组合数"))
print("-" * 78)
for label, v in [("零（未检出 / 真为零）", 0.0),
                 ("低表达 0.5", 0.5),
                 ("中表达 4.0", 4.0),
                 ("高表达 32.0", 32.0)]:
    n = distinct_inputs(v)
    print("%-30s %-16.1f %d" % (label, v, n))

print()
print("关键对比：三个不同的靶点落在同一个背景里，都在 NTC 中表达为零")
targets = ["GENE_A", "GENE_B", "GENE_C"]
seen = []
for t in targets:
    # 模型在该位置能看到的东西：(表达值=0, 标记=1)
    seen.append((0.0, 1))
print("  靶点名        ", "  ".join("%-10s" % t for t in targets))
print("  模型看到的输入", "  ".join("%-10s" % str(s) for s in seen))
print("  去重后的个数  ", len(set(seen)), "  ← 三个靶点，模型看到的是同一个东西")
print()
print("这意味着：靶点身份必须由「该基因在序列中的位置 + 它的表达值」联合携带。")
print("表达值为零时，位置是唯一剩下的线索——而位置只是一个索引，")
print("它是否能表达「这个基因是什么」完全取决于基因嵌入 emb_g(位置) 有没有学到语义。")
print()
print("补一条边界：这不是说 scGPT 一定不能用。它说明的是——")
print("  (a) 该接口对「低表达 / 零表达靶点」没有专门设计；")
print("  (b) 要修它，正确做法是给靶点一个独立编码（学靶点嵌入 / 用基因嵌入当靶点向量），")
print("      而不是加大模型或加数据。")

## 单元 3｜机制实验（合成数据）：B1 吃掉共享分量，Γ 被噪声淹没这是本课最重要的一个单元。课文 §6.4 第二层说：> 均值基线已经吃掉了共享分量的大头……一个基础模型要赢，必须证明它的额外收益来自> 背景特异分量 Γ——而 Γ 恰恰是最稀疏、最难学、也最容易被噪声淹没的那一项。**这个单元用合成数据把它演示出来。** 合成数据里我们**知道真值**，所以能算出每个方法的分数是「真的学到了 Γ」还是「拟合了噪声」。数据模型（按 [R3] `decomposition/anova.py:3` 的四分量口径）：```delta[c, t, g] = mu + alpha_c[g] + beta_t[g] + gamma[c, t, g] + eps```三个预测器：- **B0**：只给 NTC 基线，不预测任何扰动效应 → 预测 `0`- **B1**：跨背景全局 target delta → 预测 `beta_t`（训练侧中位数）- **B1+Γ**：假设有人**完美知道** Γ（上界，现实中拿不到）

In [ ]:
# ---- 合成数据：背景 C 个、靶点 T 个、基因 G 个 ----
C, T, G = 6, 40, 300
rng = np.random.default_rng(7)

def make_panel(gamma_scale, eps_scale, seed=0):
    r = np.random.default_rng(seed)
    # 共享分量 beta：跨背景不变，通常是「大」的
    beta = r.normal(0, 1.0, (T, G))
    # 背景特异分量 gamma：每个 (c,t) 一份残差，用 gamma_scale 控制能量
    gamma = r.normal(0, gamma_scale, (C, T, G))
    # 计数噪声
    eps = r.normal(0, eps_scale, (C, T, G))
    delta = beta[None, :, :] + gamma + eps
    return beta, gamma, delta

def pearson_per_t(a, b):
    """逐靶点 Pearson，再取均值。

    a/b 形状 (T, G)：留出背景上的 (T, G) 张量。
    """
    out = []
    for t in range(a.shape[0]):
        x, y = a[t].ravel(), b[t].ravel()
        if x.std() < 1e-12 or y.std() < 1e-12:
            out.append(0.0)
        else:
            out.append(float(np.corrcoef(x, y)[0, 1]))
    return float(np.mean(out))

def evaluate(gamma_scale, eps_scale, seed):
    beta, gamma, delta = make_panel(gamma_scale, eps_scale, seed)
    # 训练侧（前 C-1 个背景）估计 beta，用稳健中位数
    beta_hat = np.median(delta[: C - 1], axis=0)          # (T, G)
    # 留出背景（第 C 个）的真值
    d_out = delta[C - 1]                                   # (T, G)
    pred_B1 = np.broadcast_to(beta_hat, (T, G))
    pred_B1G = beta_hat + gamma[C - 1]                     # 假设完美知道 Γ
    return pearson_per_t(d_out, pred_B1), pearson_per_t(d_out, pred_B1G)

print("%-12s %-12s %10s %10s %10s" % ("Γ 强度", "噪声", "B0", "B1", "B1+完美Γ"))
print("-" * 60)
for gs in (0.1, 0.5, 1.0, 2.0):
    for es in (0.5, 1.0):
        s_b0 = 0.0  # B0 输出恒零，与任何东西的 Pearson 都是 0
        s_b1, s_b1g = evaluate(gs, es, seed=11)
        print("%-12.1f %-12.1f %10.4f %10.4f %10.4f" % (gs, es, s_b0, s_b1, s_b1g))
print()
print("读法：")
print("  · Γ 很弱（0.1）时，B1 ≈ B1+完美Γ —— 知道 Γ 也没用，共享分量就是全部。")
print("  · Γ 变强时，B1 的分数反而下降（因为 Γ 成了噪声源），而完美Γ的上界在上升。")
print("  · 这就是「基础模型的机会窗口」：Γ 的能量必须显著大于噪声，高容量模型才有空间。")

In [ ]:
# ---- 把「机会窗口」画出来：Γ 能量 vs 噪声，B1 与上界的比值 ----
grid = np.array([0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2])
loss_ratio = []
for gs in grid:
    s_b1, s_b1g = evaluate(gs, 1.0, seed=11)
    loss_ratio.append(s_b1 / s_b1g if abs(s_b1g) > 1e-9 else 0.0)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(grid, loss_ratio, "o-", lw=2, color="#c0392b", label="B1 / (B1 + 完美Γ)")
ax.axhline(1.0, ls="--", lw=1, color="#7f8c8d")
ax.text(grid[0], 1.01, "B1 已经吃完（Γ 无价值）", fontsize=9, color="#7f8c8d")
ax.set_xscale("log", base=2)
ax.set_xlabel("背景特异分量 Γ 的强度（对数刻度）", fontsize=11)
ax.set_ylabel("B1 分数 / 完美 Γ 的上界分数", fontsize=11)
ax.set_title("共享分量被 B1 吃掉：什么时候高容量模型还有空间", fontsize=12)
ax.set_ylim(0, 1.15)
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
fig.tight_layout()

# 图只作本地诊断，输出到 Git 已忽略的 output/，不写进课件 assets
fig_dir = Path("../output/tmp")
fig_dir.mkdir(parents=True, exist_ok=True)
out_png = fig_dir / "nb08-beta-gamma-window.png"
fig.savefig(out_png, bbox_inches="tight")
plt.close(fig)
print("已写出", out_png, "（本地诊断图，Git 忽略）")
print()
print("%-10s %s" % ("Γ 强度", "B1 / 上界"))
for gs, r in zip(grid, loss_ratio):
    print("%-10.2f %.4f" % (gs, r))
print()
print("注意：这是合成数据，演示的是机制不是任何真实数据集的结论。")
print("真实面板上应当用 anova.py 的口径从数据里估出 beta_frac，见单元 4。")

## 单元 4｜按 transferable fraction 给靶点分档[R3] `decomposition/anova.py:68-72` 为每个扰动单独算一个比值：$$\text{beta\_frac}(t) = \frac{\operatorname{Var}(\beta_t)}{\operatorname{Var}(\beta_t) + \operatorname{mean}_c \operatorname{Var}(\gamma_{c,t})}$$这个比值高 = 共享分量占主导 = B1 就能做好；比值低 = 有 Γ 的空间 = 才值得投算力模型。**这在真实面板上的直接做法**（对应课文 §9.3 的诊断题）：在训练背景上跑一次四分量分解，对每个靶点算 `beta_frac`，然后分档。零覆盖靶点（在训练背景里一次都没被测过）单独拎出来——它们只能走 B0。

In [ ]:
def beta_frac(beta_t, gamma_ct):
    """[R3] decomposition/anova.py:68-72 的直接转写。
    beta_t: (G,) 共享分量；gamma_ct: (C, G) 各背景的交互项。
    返回 (比值, Var(beta), mean_c Var(gamma))。
    """
    b_var = float(np.var(beta_t))
    g_var = float(np.mean([np.var(gamma_ct[c]) for c in range(gamma_ct.shape[0])]))
    return b_var / (b_var + g_var + 1e-30), b_var, g_var

# 用合成面板算一遍完整流程。
# 关键：Γ 的强度**因靶点而异**（真实面板里不同靶点的可迁移性本来就不同），
# 否则全部靶点的 beta_frac 会挤在同一段，分档就没有意义。
beta, gamma, delta = make_panel(gamma_scale=0.8, eps_scale=1.0, seed=11)
# 给每个靶点一个不同的 Γ 倍率，覆盖从「几乎全可迁移」到「几乎全是背景特异」
per_t = np.exp(rng.normal(0, 1.1, T))          # 对数正态，长尾
per_t = per_t / per_t.mean()                    # 归一，保持总体能量
gamma = gamma * per_t[None, :, None]

rows = []
for t in range(T):
    frac, bv, gv = beta_frac(beta[t], gamma[:, t, :])
    rows.append((t, frac, bv, gv))

rows.sort(key=lambda r: r[1])
print("%-8s %-12s %-12s %-12s %s" % ("靶点", "beta_frac", "Var(beta)", "mean Var(gamma)", "分档"))
print("-" * 78)

def bucket(f):
    if f >= 0.70:
        return "高：B1 就够，别投算力"
    if f >= 0.50:
        return "中：可投，但要过噪声地板"
    return "低：Γ 主导，最值得投"

for t, frac, bv, gv in rows[:3] + rows[len(rows)//2-1:len(rows)//2+1] + rows[-3:]:
    print("%-8d %-12.4f %-12.4f %-12.4f %s" % (t, frac, bv, gv, bucket(frac)))

cnt = Counter(bucket(f) for _, f, _, _ in rows)
print()
print("分档统计（阈值 0.70 / 0.50）：", dict(cnt))
print()
print("阈值不是物理常数，是从「B1 在留出背景上的分数是否已接近上界」反推出来的。")
print("用法：在公开真值面板上跑 anova，对每个靶点同时记录 beta_frac 和 B1 的实际分数，")
print("      再按分数曲线找拐点，而不是照搬这里的 0.70 / 0.50。")
print()
print("决策规则（课文 §7 决策二 / §9.3）：")
print("  1. 先剔除在可用训练数据上零覆盖的靶点 —— 它们没有 Var(beta) 可估，只能走 B0。")
print("  2. 剩下的按 beta_frac 分档，把算力集中在中低档。")
print("  3. 任何在中高档靶点上的「改进」，先过 L1-02 的噪声地板再谈。")

In [ ]:
# ---- 模拟本赛 300 个靶点的分配决策 ----
N_TARGETS = 300
r = np.random.default_rng(2026)

# 假设：约 1/4 的靶点在可用训练面板上零覆盖（课文 §9.3 第 3 题的情形）
zero_cov = 0.25
frac_low = 0.35     # 中低档（值得投）
frac_mid = 0.30     # 中档
frac_high = 1 - zero_cov - frac_low - frac_mid   # 高档

n_zero = int(N_TARGETS * zero_cov)
n_low = int(N_TARGETS * frac_low)
n_mid = int(N_TARGETS * frac_mid)
n_high = N_TARGETS - n_zero - n_low - n_mid

print("本赛 300 个靶点的算力分配（工程假设的参数化演示）")
print("-" * 60)
print("%-28s %6s %10s" % ("档位", "靶点数", "该走什么路线"))
print("-" * 60)
print("%-28s %6d %10s" % ("零覆盖（训练侧一次未测）", n_zero, "B0"))
print("%-28s %6d %10s" % ("低 beta_frac（Γ 主导）", n_low, "State / B3"))
print("%-28s %6d %10s" % ("中 beta_frac", n_mid, "先 B1，过地板再升级"))
print("%-28s %6d %10s" % ("高 beta_frac（β 主导）", n_high, "B1 即可"))
print("-" * 60)
print()
print("注意这个分配表里的数字是工程假设（用于说明决策结构），不是本赛的真实统计。")
print("真实分档必须先在一份有真值的公开面板上跑 anova，见课文 §8 检查表。")

## 单元 5｜把结论带回课文跑完请回答课文 §8 检查表的四问，并把答案抄进自己的实验记录：1. **scGPT 的规模该怎么引用？** —— 为什么词表项 48,292 × 512 不能被并进一个「总参数量」？2. **靶点表达为零时模型还剩什么？** —— 单元 2 的数法给了什么答案？修它应该改什么？3. **共享分量占 95% 的靶点值得投吗？** —— 用单元 3 的机会窗口图回答，判据是什么？4. **AIDO Cell 的哪一类留出离本赛最近？** —— 差在哪一个轴上（见课文 §5.3）？**三件本课明确没做的事**（写进记录，不要当成已验证）：- 没有下载任何权重、没有运行 scGPT / AIDO Cell；- 没有复现 `[P3]`/`[P4]` 的基准数值，单元 3/4/5 用的是**合成数据**，演示机制不是复现结论；- 没有给出 AIDO Cell 的参数量与语料规模 —— 报告里没有，本课就不填。**下一步**：带着这里的 `beta_frac` 分档口径进 [L2-04 简单基线与效应迁移族](../docs/lessons/README.md)〔待写〕，它会给 B0–B3 的实测结果；本课只负责给出「该往哪儿投」的判据。